# WebNeuro data QC pipeline

Logic to check edge cases for tasks in WebNeuro for ACE-D AIM 3 and AIM 2

The current issue with WebNeuro is that missingness for certain tasked are labelled as 0 instead of missing for certain variables. For example, when participant RT looks like missing because the task time-out it labelled as missing, but the "vi_sco1" number of stimuli was correctly selected was labelled as 0 instead of missing. What we want to do is to streamline the notion of missing data vs timeoput with effort so that it is consistent across all tasks.The other is that to correct variables when 0 is being used to denote missingness. Another example is N-Back RT is missing but have 0 false positive errors (because they did not press anything) resulting in high normed scores.

1. For maze task
  Time cannot be 0, we check for 0 edge cases
  - emzcompk: the total time to complete the maze
  - emzinitk: Path learning time describe how long it takes for partiicpant to learn before complete the maze twice
  Error cannot be 0, we chheck 0 for edge cases. Check against the healthy reference number of errors.
  - emzerrk: the total errors of all types 
  - emzoverk: Overrun errors are specifically when subjects continue pressing in the same direction of movement through the maze instead of turning where required."
  - emztrlsk: Number of trials taken to complete the maze correctly twice. Always greater > 2 times and get the healthy reference range for this.

2. GNG task
 Time cannot be 0 
 - g2avrtk: the average reaction time between stimuli flashing and spacebar press across all correctly performed trials. 
  The reaction time cannot be greater than 2.7 seconds (the interval between stimuli)
 - g2errk: total errors sum of false positive and false negative. Cannot be 0 erros.
 - g2fnk: total of false negative. Cannot be 0 and < 126 and Sum of g2fnk + g2fpk = g2errk
 - g2fpk: total of false positive . Cannot be 0 and < 42

3. Stroop task
 - vcrtne: the average reaction time for correct response. Cannot be 0 and less than 30, unsure whether the task time out at 20 or 30 seconds
 - vi_sco1: Number of stimuli was correctly selected. Cannot be < 0

In [1]:
import glob
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Logics to check
# Denote impairment timeout with efforts versus missing time-out without efforts
# Task order: 
# 1. Motor tapping,
# 2. Verbal memory
# 3. Emotion Recognition
# 4. Digital Span
# 5. Verbal Interference / Stroop
# 6. Switch of Attention
# 7. Go-NoGo
# 8. Delayed Memory
# 9. Emotion Identification
# 10. N-Back
# 11. Maze


# 1. For maze task
#   Time cannot be 0, we check for 0 edge cases. The task time out at 16 minutes
#   - emzcompk: the total time to complete the maze. Note emzcompk > emzinitk and missing if timeout
#   - emzinitk: Path learning time describe how long it takes for partiicpant to learn before complete the maze correctly twice. "Path learning time` or time taken to the end of the final trial with at least one error- just before successfully completing the maze without error twice consecutively ",
#   Error cannot be 0, we chheck 0 for edge cases. Check against the healthy reference number of errors.
#   - emzerrk: the total errors of all types made across all trials taken to complete the maze without error twice consecutively. It is impossible to have emzerrk = 0
#   - emzoverk: "Cumulative number of `overrun` errors made across all trials taken to complete the maze without error twice consecutively. Overrun errors are specifically when subjects continue pressing in the same direction of movement through the maze instead of turning where required.". Impossible to have 0 
#   - emztrlsk: Number of trials taken to complete the maze correctly twice. Always greater > 2 times and get the healthy reference range for this.
#   For the error variable, missing if the task timeout which the participant did not do anything because no error were registerd. However, the task can timeout and partiicpant did have so many errors they couldnt finish the task in time.

# 2. GNG task
#  Time cannot be 0 
#  - g2avrtk: the average reaction time between stimuli flashing green and spacebar press across all correctly performed trials. This can not be 0 and greater 2.7 seconds because duration between each stimuli flasing is 2.7 seconds
#   The reaction time cannot be greater than 2.7 seconds (the interval between stimuli)
#  - g2errk: There are 168 trials for the go no go task, total errors sum of false positive and false negative. The total erros can be 0 if participant did it correctly, and value is labelled missing when no response made to any stimuli.it should not be 0 when it's missing
#  - g2fnk: total of false negative. The total green stimuli appearing 126 time, 0 <= value <= 126 and Sum of g2fnk + g2fpk = g2errk  false negative is when it's green they not pressing. The total false negative erros can be 0 if participant did all the green stimuli correctly, and value labelled missing when no response made to any stimuli. It should not be 0 if missing
#  - g2fpk: total of false positive . 0 <= value <= 42 The total red stimuli appearing 48 time, 0 <= value <= 48 and Sum of g2fnk + g2fpk = g2errk  false positive is when it's red they press. The total false positive erros can be 0 if participant did all the red stimuli correctly, and value labelled missing when no response made to any stimuli. It should not be 0 if missing
# For the error, the g2errk = g2fnk + g2fpk,

# 3. Stroop task
#  - vcrtne: the average reaction time for correct responses for color-words was correctly selected (ignoring the font-color of the word). Cannot be 0 and less than 30. Task timeout at 30s
#  - vi_sco1:"Number of stimuli for which the name of the color-word was correctly selected (ignoring the font-color of the word) in the time limit of 30 seconds (i.e a higher score reflects greater speed)". It should be labelled as missing if no response, 
#             value cannot be < 0, also if vi_sco1 is 0 which suggesting participant did not get anything correct the vcrtne should be missing/blank and not 0 
#  - vcrtne2: the average reaction time for correct responses for which the font-color of the color-word was correctly selected (ignoring the name of the word). Cannot be 0 and less than 30. Task timeout at 30s
#  - vi_sco2: "Number of stimuli for which the font-color of the color-word was correctly selected (ignoring the name of the word) in the time limit of 30 seconds (i.e a higher score reflects greater speed)".It should be labelled as missing if no response
#             value cannot be < 0,  also if vi_sco2 is 0 which suggesting participant did not get anything correct the vcrtne should be missing/blank and not 0   

# 4. digital span task
#  - digitot: total number of correct trials over 14 trials. 0 <= Value <= 14, missing if participant did not do the task should not be label as 0.
#  - digitsp: The longest digit span that participant can remember after seeing them and entered on a keypad in correct order. 
#             0,3 =< Value =< 9, participant can get 0 if they did not get any sequence correctly. 
#             If this score is 0 digittot has to be 0, if this score is missing digitot has to be missing.

# 5. Nback task
# Current issue - wmrtk is missing we dont know if they did the task and get everything wrong or they let its timeout. When the False Positive 
# Current issues:



In [ ]:
# Data input
WN_COMPOSITE_GROUPS = [
    ("maze_composite", ["emzcompk", "emzinitk", "emzerrk", "emzoverk", "emztrlsk"]),
    ("gng_composite", ["g2avrtk", "g2errk", "g2fnk", "g2fpk"]),
    ("stroopw_composite", ["vcrtne", "vi_sco1"]),
    ("stroopc_composite", ["vcrtne2", "vi_sco2"]),
    ("swoa_composite", ["esoadur2", "scavr0t2", "esoaerr2"]),
    ("digit_composite", ["digitot", "digitsp"]),
]

aim2_df = pd.read_csv("/Users/ttan306/Desktop/ACE-D/AIM_2/ACEDStudy2PRIMARY-WebNeuroData_DATA_2026-08-10_1902.csv",delimiter=",")
aim2_df_t1 = aim2_df[aim2_df['redcap_event_name'] == 't1_arm_1']


/var/folders/hl/m7mhy4qd0zq7zb20bqfqv5zh0000gq/T/ipykernel_46291/1747437581.py:2: DtypeWarning: Columns (0: record_id, 1: issue_note, 2: esoadur2, 3: esoaerr2, 4: scavr0t2) have mixed types. Specify dtype option on import or set low_memory=False.
  aim2_df = pd.read_csv("/Users/ttan306/Desktop/ACE-D/AIM_2/ACEDStudy2PRIMARY-WebNeuroData_DATA_2026-08-10_1902.csv",delimiter=",")


Index(['record_id', 'redcap_event_name', 'study_id', 'aced_site', 'wn_alias',
       'wn_timepoint', 'wn_datecompete', 'issue_yn', 'issue_note', 'gender',
       ...
       'emzcompk_norm', 'emzerrk', 'emzerrk_norm', 'emzinitk', 'emzinitk_norm',
       'emzoverk', 'emzoverk_norm', 'emztrlsk', 'emztrlsk_norm',
       'webneuro_data_complete'],
      dtype='str', length=141)

In [21]:
aim2_df_t1.columns.to_list()

['record_id',
 'redcap_event_name',
 'study_id',
 'aced_site',
 'wn_alias',
 'wn_timepoint',
 'wn_datecompete',
 'issue_yn',
 'issue_note',
 'gender',
 'testdate',
 'age',
 'tdomnk',
 'tdomnk_norm',
 'tdomsdk',
 'tdomsdk_norm',
 'chlrrtav',
 'chlrrtav_norm',
 'ctmrec1',
 'ctmrec1_norm',
 'ctmrec2',
 'ctmrec2_norm',
 'ctmrec3',
 'ctmrec3_norm',
 'ctmsco13',
 'ctmsco13_norm',
 'getcpa',
 'getcpa_norm',
 'getcpd',
 'getcpd_norm',
 'getcpf',
 'getcpf_norm',
 'getcph',
 'getcph_norm',
 'getcpn',
 'getcpn_norm',
 'getcps',
 'getcps_norm',
 'getcrta',
 'getcrta_norm',
 'getcrtd',
 'getcrtd_norm',
 'getcrtf',
 'getcrtf_norm',
 'getcrth',
 'getcrth_norm',
 'getcrtn',
 'getcrtn_norm',
 'getcrts',
 'getcrts_norm',
 'gettrta',
 'gettrta_norm',
 'gettrtd',
 'gettrtd_norm',
 'gettrtf',
 'gettrtf_norm',
 'gettrth',
 'gettrth_norm',
 'gettrtn',
 'gettrtn_norm',
 'gettrts',
 'gettrts_norm',
 'digitot',
 'digitot_norm',
 'digitsp',
 'digitsp_norm',
 'vi_difrt',
 'vi_difrt_norm',
 'vcrtne2',
 'vcrtne2_no

In [15]:

WN_COMPOSITE_GROUPS = [
    ("maze_composite", ["emzcompk", "emzinitk", "emzerrk", "emzoverk", "emztrlsk"]),
    ("gng_composite", ["g2avrtk", "g2errk", "g2fnk", "g2fpk"]),
    ("stroopw_composite", ["vcrtne", "vi_sco1"]),
    ("stroopc_composite", ["vcrtne2", "vi_sco2"]),
    ("swoa_composite", ["esoadur2", "scavr0t2", "esoaerr2"]),
    ("digit_composite", ["digitot", "digitsp"]),
]

In [2]:
aim3_df = pd.read_csv("/Users/ttan306/Desktop/ACE-D/AIM_3/ACEDAIM3Primary-WebNeuroData_DATA_2026-08-12_1033.csv",delimiter=",")
aim3_df

,record_id,redcap_event_name,redcap_repeat_instrument,redcap_repeat_instance,study_id,wn_alias,wn_timepoint,wn_datecompete,issue_yn,issue_note,...,emzcompk_norm,emzerrk,emzerrk_norm,emzinitk,emzinitk_norm,emzoverk,emzoverk_norm,emztrlsk,emztrlsk_norm,webneuro_data_complete
0,2,enroll_arm_1,NaN,NaN,3SDHX99996,75RYUM,1.0,2026-03-24,NaN,NaN,...,1.313,20.0,0.713,53698.0,1.416,7.0,0.691,4.0,1.288,2
1,2,scan1_arm_1,NaN,NaN,NaN,NaN,2.0,2026-05-08,NaN,NaN,...,0.991,40.0,0.167,82951.2,1.091,12.0,0.448,7.0,0.581,2
2,2,tx3_arm_1,NaN,NaN,NaN,NaN,4.0,2026-07-09,1.0,The participant was not sent Early Treatment W...,...,1.543,5.0,1.123,35749.7,1.616,1.0,0.983,6.0,0.817,2
3,2,scan2_arm_1,NaN,NaN,NaN,NaN,3.0,2026-07-09,NaN,NaN,...,1.402,20.0,0.713,48480.0,1.474,5.0,0.788,5.0,1.052,2
4,3,enroll_arm_1,NaN,NaN,3SJLX99997,VS4HTI,1.0,2026-03-11,NaN,NaN,...,0.558,38.0,0.361,189118.0,0.565,8.0,0.462,6.0,0.975,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,614,enroll_arm_1,NaN,NaN,3SAMX99860,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
88,617,enroll_arm_1,NaN,NaN,3SJLX99864,NE5MAF,1.0,2026-07-28,NaN,NaN,...,-0.523,43.0,-0.049,236916.0,-0.526,22.0,-0.239,15.0,-1.351,2
89,624,enroll_arm_1,NaN,NaN,3UJGX99983,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
90,638,enroll_arm_1,NaN,NaN,3SZRX99858,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [7]:
list(aim3_df.columns)

['record_id',
 'redcap_event_name',
 'redcap_repeat_instrument',
 'redcap_repeat_instance',
 'study_id',
 'wn_alias',
 'wn_timepoint',
 'wn_datecompete',
 'issue_yn',
 'issue_note',
 'gender',
 'testdate',
 'age',
 'wn_timepoint_imaging',
 'tdomnk',
 'tdomnk_norm',
 'tdomsdk',
 'tdomsdk_norm',
 'chlrrtav',
 'chlrrtav_norm',
 'ctmrec1',
 'ctmrec1_norm',
 'ctmrec2',
 'ctmrec2_norm',
 'ctmrec3',
 'ctmrec3_norm',
 'ctmsco13',
 'ctmsco13_norm',
 'getcpa',
 'getcpa_norm',
 'getcpd',
 'getcpd_norm',
 'getcpf',
 'getcpf_norm',
 'getcph',
 'getcph_norm',
 'getcpn',
 'getcpn_norm',
 'getcps',
 'getcps_norm',
 'getcrta',
 'getcrta_norm',
 'getcrtd',
 'getcrtd_norm',
 'getcrtf',
 'getcrtf_norm',
 'getcrth',
 'getcrth_norm',
 'getcrtn',
 'getcrtn_norm',
 'getcrts',
 'getcrts_norm',
 'gettrta',
 'gettrta_norm',
 'gettrtd',
 'gettrtd_norm',
 'gettrtf',
 'gettrtf_norm',
 'gettrth',
 'gettrth_norm',
 'gettrtn',
 'gettrtn_norm',
 'gettrts',
 'gettrts_norm',
 'digitot',
 'digitot_norm',
 'digitsp',
 'dig

In [ ]:
# Participant who was randomized
# "3SJRX99889",
# "3UNGE12046",
# "3SVFE99893",
# "3SJUX99876",
# "3UKGX99993",
# "3USIE11714",
# "3UOVE11107",
# "3UASX99994",
# "3SEDX99938",
# "3URJE11266",
# "3SRRX99954",
# "3SDHX99996,

In [22]:
aim3_df.dtypes

record_id                     int64
redcap_event_name               str
redcap_repeat_instrument    float64
redcap_repeat_instance      float64
study_id                        str
                             ...   
emzoverk                    float64
emzoverk_norm               float64
emztrlsk                    float64
emztrlsk_norm               float64
webneuro_data_complete        int64
Length: 143, dtype: object

In [32]:
record_to_study = (
    aim3_df.dropna(subset=["study_id"])
    .drop_duplicates(subset="record_id")
    .set_index("record_id")["study_id"]
)
record_to_study

record_id
2      3SDHX99996
3      3SJLX99997
14     3SCCX99985
47     3SDDX99955
48     3SRRX99954
93     3SJVX99945
105    3UCLE10057
110    3SJBX99943
111    3SPLX99942
162    3UCIE10164
178    3SEDX99938
189    3UEDE10008
195    3UCCE10468
197    3UMGE10643
210    3UAAE11125
213    3URJE11266
224    3SCBE99930
225    3UJGE10351
284    3UDPE11193
303    3UASX99994
311    3UMRE11406
316    3UJHE10638
336    3SLSX99905
365    3UOVE11107
375    3UKGX99993
377    3SSBX99897
390    3SVFE99893
408    3SJRX99889
410    3SCYX99886
417    3UNGE12046
453    3UMVX99990
460    3UYJX99991
483    3USIE11714
496    3UBME10445
498    3SNBX99879
526    3SJUX99876
527    3SAVX99875
539    3SBSX99874
562    3SAMX99871
564    3SCGX99867
598    3SCCX99862
599    3SCTX99863
602    3SAVX99861
612    3UJLX99986
614    3SAMX99860
617    3SJLX99864
624    3UJGX99983
638    3SZRX99858
721    3UJMX99975
Name: study_id, dtype: str

In [39]:
aim3_screen_df = aim3_df.loc[
    (aim3_df["redcap_event_name"] == "enroll_arm_1")][
        [   "record_id", "study_id","redcap_event_name",
            "emzcompk", "emzinitk", "emzerrk", "emzoverk", "emztrlsk",
            "g2avrtk", "g2errk", "g2fnk", "g2fpk",
            "vcrtne", "vi_sco1",
            "vcrtne2", "vi_sco2",
            "esoadur2", "scavr0t2", "esoaerr2",
            "digitot", "digitsp"
         ]
        ]
aim3_screen_df

,record_id,study_id,redcap_event_name,emzcompk,emzinitk,emzerrk,emzoverk,emztrlsk,g2avrtk,g2errk,...,g2fpk,vcrtne,vi_sco1,vcrtne2,vi_sco2,esoadur2,scavr0t2,esoaerr2,digitot,digitsp
0,2,3SDHX99996,enroll_arm_1,76623.0,53698.0,20.0,7.0,4.0,304.650,66.0,...,0.0,3082.330,6.0,2000.89,9.0,33249.0,1329.96,0.0,9.0,8.0
4,3,3SJLX99997,enroll_arm_1,226369.0,189118.0,38.0,8.0,6.0,208.372,4.0,...,4.0,1660.350,10.0,2324.73,7.0,86562.1,3462.48,2.0,7.0,6.0
6,14,3SCCX99985,enroll_arm_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,47,3SDDX99955,enroll_arm_1,116959.0,103526.0,32.0,14.0,10.0,96.512,1.0,...,0.0,802.158,19.0,2303.25,8.0,37818.0,1512.72,0.0,7.0,6.0
8,48,3SRRX99954,enroll_arm_1,185285.0,162350.0,56.0,23.0,12.0,213.575,7.0,...,5.0,1006.180,16.0,3673.30,5.0,62882.5,2515.30,4.0,8.0,8.0
12,93,3SJVX99945,enroll_arm_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,105,3UCLE10057,enroll_arm_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,110,3SJBX99943,enroll_arm_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,111,3SPLX99942,enroll_arm_1,175398.0,108650.0,16.0,3.0,4.0,186.904,3.0,...,2.0,978.176,17.0,1316.31,13.0,58390.0,2335.60,2.0,8.0,7.0
16,162,3UCIE10164,enroll_arm_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
aim3_baseline_df = aim3_df.loc[
    (aim3_df["redcap_event_name"] == "scan1_arm_1")][
        [   "record_id", "study_id","redcap_event_name",
            "emzcompk", "emzinitk", "emzerrk", "emzoverk", "emztrlsk",
            "g2avrtk", "g2errk", "g2fnk", "g2fpk",
            "vcrtne", "vi_sco1",
            "vcrtne2", "vi_sco2",
            "esoadur2", "scavr0t2", "esoaerr2",
            "digitot", "digitsp"
         ]
        ]
aim3_baseline_df["study_id"] = aim3_baseline_df["record_id"].map(record_to_study)
aim3_baseline_df

,record_id,study_id,redcap_event_name,emzcompk,emzinitk,emzerrk,emzoverk,emztrlsk,g2avrtk,g2errk,...,g2fpk,vcrtne,vi_sco1,vcrtne2,vi_sco2,esoadur2,scavr0t2,esoaerr2,digitot,digitsp
1,2,3SDHX99996,scan1_arm_1,108340.0,82951.2,40.0,12.0,7.0,129.737,8.0,...,8.0,1075.00,15.0,1435.88,12.0,32885.8,1315.43,1.0,8.0,6.0
5,3,3SJLX99997,scan1_arm_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,48,3SRRX99954,scan1_arm_1,125179.0,99113.9,43.0,13.0,7.0,270.854,3.0,...,2.0,2226.52,8.0,2567.64,7.0,48955.2,1958.21,4.0,8.0,7.0
18,178,3SEDX99938,scan1_arm_1,320656.0,277091.0,21.0,12.0,8.0,320.873,0.0,...,0.0,1586.55,11.0,2844.67,6.0,43182.0,1727.28,0.0,8.0,6.0
26,213,3URJE11266,scan1_arm_1,216868.0,192656.0,52.0,7.0,9.0,169.736,1.0,...,0.0,1276.46,13.0,1588.55,11.0,36230.0,1449.20,2.0,12.0,9.0
33,303,3UASX99994,scan1_arm_1,155626.0,32635.0,16.0,3.0,3.0,159.771,4.0,...,4.0,1266.86,13.0,1412.17,12.0,2781.5,2781.50,0.0,12.0,9.0
39,336,3SLSX99905,scan1_arm_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,365,3UOVE11107,scan1_arm_1,106410.0,80812.0,26.0,7.0,5.0,222.159,1.0,...,1.0,1162.00,13.0,1203.23,13.0,39815.0,1592.60,1.0,10.0,8.0
47,375,3UKGX99993,scan1_arm_1,174497.0,141267.0,26.0,8.0,8.0,271.299,5.0,...,4.0,1429.57,12.0,1609.75,11.0,73394.7,2935.79,0.0,8.0,9.0
52,390,3SVFE99893,scan1_arm_1,281697.0,248621.0,19.0,7.0,10.0,216.402,2.0,...,1.0,1721.14,10.0,2126.72,8.0,44948.8,1797.95,0.0,5.0,5.0


In [58]:
aim3_baseline_df['study_id'].unique()

<StringArray>
['3SDHX99996', '3SJLX99997', '3SRRX99954', '3SEDX99938', '3URJE11266',
 '3UASX99994', '3SLSX99905', '3UOVE11107', '3UKGX99993', '3SVFE99893',
 '3SJRX99889', '3UNGE12046', '3USIE11714', '3UBME10445', '3SJUX99876']
Length: 15, dtype: str

In [ ]:
randomized_pt = [
"3SJRX99889",
"3UNGE12046",
"3SVFE99893",
"3SJUX99876",
"3UKGX99993",
"3USIE11714",
"3UOVE11107",
"3UASX99994",
"3SEDX99938",
"3URJE11266",
"3SRRX99954",
"3SDHX99996"]

aim3_baseline_df['study_id'].isin(randomized_pt)

1      True
5     False
9      True
18     True
26     True
33     True
39    False
43     True
47     True
52     True
56     True
61     True
67     True
71    False
76     True
Name: study_id, dtype: bool

In [104]:
aim3_screen_df_filtered = aim3_screen_df.loc[aim3_screen_df['study_id'].isin(randomized_pt)]
aim3_baseline_df_filtered = aim3_baseline_df.loc[aim3_baseline_df['study_id'].isin(randomized_pt)]

In [131]:
group_vars

['digitot', 'digitsp']

In [136]:
group_vars

['digitot', 'digitsp']

In [141]:
import plotly.express as px
import plotly.graph_objects as go
import os

WN_COMPOSITE_GROUPS = [
    ("maze_composite", ["emzcompk", "emzinitk", "emzerrk", "emzoverk", "emztrlsk"]),
    ("gng_composite", ["g2avrtk", "g2errk", "g2fnk", "g2fpk"]),
    ("stroopw_composite", ["vcrtne", "vi_sco1"]),
    ("stroopc_composite", ["vcrtne2", "vi_sco2"]),
    ("swoa_composite", ["esoadur2", "scavr0t2", "esoaerr2"]),
    ("digit_composite", ["digitot", "digitsp"]),
]

MS_VARS = {"emzcompk", "emzinitk", "g2avrtk", "vcrtne", "vcrtne2", "esoadur2", "scavr0t2"}
BORDER_COLOR = "#c3c2b7"
MISSING_COLOR = "#898781"
N_COLS = 3


In [146]:
# plotting raw WebNeuro Score for AIM 3
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import os


def facet_row_col(index, ncols, total):
    """Plotly's row/col selectors count rows from the bottom, so convert
    a row-major (top-to-bottom) index into that convention."""
    n_rows = -(-total // ncols)  # ceil division
    r0, c0 = index // ncols, index % ncols
    return n_rows - r0, c0 + 1


def prepare_plot_df(df, ms_vars):
    plot_df = df.rename(columns={"study_id": "participant_id"}).copy()
    for var in ms_vars:
        plot_df[var] = plot_df[var] / 1000
    return plot_df


def build_color_map(plot_df, palette=None):
    palette = palette or px.colors.qualitative.Dark24
    participant_ids = sorted(plot_df["participant_id"].dropna().unique())
    return {pid: palette[i % len(palette)] for i, pid in enumerate(participant_ids)}


def melt_group(plot_df, group_vars, ms_vars):
    long_df = plot_df.melt(
        id_vars=["participant_id"],
        value_vars=group_vars,
        var_name="measure",
        value_name="value",
    )
    long_df["measure_label"] = long_df["measure"].apply(
        lambda v: f"{v} (s)" if v in ms_vars else v
    )
    return long_df


def add_missing_markers(fig, long_df, group_vars, ms_vars, color_map, n_cols):
    for idx, var in enumerate(group_vars):
        label = f"{var} (s)" if var in ms_vars else var
        row, col = facet_row_col(idx, n_cols, len(group_vars))
        missing_pids = long_df.loc[
            (long_df["measure_label"] == label) & long_df["value"].isna(),
            "participant_id",
        ]
        if len(missing_pids) == 0:
            continue
        fig.add_trace(
            go.Scatter(
                x=missing_pids,
                y=[0] * len(missing_pids),
                mode="markers",
                marker=dict(
                    symbol="x",
                    size=9,
                    color=[color_map[pid] for pid in missing_pids],
                ),
                hovertemplate="%{x}<br>Missing<extra></extra>",
                showlegend=False,
            ),
            row=row,
            col=col,
        )


def style_composite_figure(fig, border_color):
    fig.update_xaxes(
        title_text="participant_id",
        showticklabels=False,
        showline=True,
        linewidth=1,
        linecolor=border_color,
        mirror=True,
        matches=None,
    )
    fig.update_yaxes(
        matches=None,
        showticklabels=True,
        exponentformat="none",
        showline=True,
        linewidth=1,
        linecolor=border_color,
        mirror=True,
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.update_layout(
        plot_bgcolor="#fcfcfb",
        paper_bgcolor="#fcfcfb",
        margin=dict(t=60, b=40),
        hoverlabel=dict(bgcolor="#fcfcfb", font_color="#0b0b0b", font_size=12, bordercolor=border_color),
    )


def build_composite_figure(group_name, group_vars, plot_df, ms_vars, color_map, n_cols, border_color):
    long_df = melt_group(plot_df, group_vars, ms_vars)
    measure_order = [f"{v} (s)" if v in ms_vars else v for v in group_vars]

    fig = px.strip(
        long_df,
        x="participant_id",
        y="value",
        color="participant_id",
        color_discrete_map=color_map,
        hover_data=["participant_id"],
        facet_col="measure_label",
        facet_col_wrap=n_cols,
        facet_col_spacing=0.06,
        facet_row_spacing=0.15,
        category_orders={"measure_label": measure_order},
        title=group_name,
        height=450,
    )
    add_missing_markers(fig, long_df, group_vars, ms_vars, color_map, n_cols)
    style_composite_figure(fig, border_color)
    return fig


def write_figures_html(figs, output_path):
    figs_html = [
        fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False)
        for i, fig in enumerate(figs)
    ]
    spaced = [f'<div style="margin-bottom:60px;">{h}</div>' for h in figs_html]
    with open(output_path, "w") as f:
        f.write("<html><head><meta charset='utf-8'></head><body>\n")
        f.write("\n".join(spaced))
        f.write("\n</body></html>")


def generate_composite_report(df, group_defs, ms_vars, output_path, n_cols=3, border_color="#c3c2b7", color_map=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plot_df = prepare_plot_df(df, ms_vars)
    color_map = color_map or build_color_map(plot_df)
    figs = [
        build_composite_figure(group_name, group_vars, plot_df, ms_vars, color_map, n_cols, border_color)
        for group_name, group_vars in group_defs
    ]
    write_figures_html(figs, output_path)
    return figs



In [147]:
color_map = build_color_map(prepare_plot_df(aim3_screen_df_filtered, MS_VARS))

generate_composite_report(
    aim3_screen_df_filtered, WN_COMPOSITE_GROUPS, MS_VARS,
    "plots/wn_composites.html", color_map=color_map,
)
generate_composite_report(
    aim3_baseline_df_filtered, WN_COMPOSITE_GROUPS, MS_VARS,
    "plots/wn_baseline_composites.html", color_map=color_map,
)


[Figure({
     'data': [{'alignmentgroup': 'True',
               'boxpoints': 'all',
               'fillcolor': 'rgba(255,255,255,0)',
               'hoveron': 'points',
               'hovertemplate': 'participant_id=%{x}<br>measure_label=emzcompk (s)<br>value=%{y}<extra></extra>',
               'legendgroup': '3SDHX99996',
               'line': {'color': 'rgba(255,255,255,0)'},
               'marker': {'color': '#2E91E5'},
               'name': '3SDHX99996',
               'offsetgroup': '3SDHX99996',
               'orientation': 'v',
               'pointpos': 0,
               'showlegend': True,
               'type': 'box',
               'x': array(['3SDHX99996'], dtype=object),
               'x0': ' ',
               'xaxis': 'x4',
               'y': {'bdata': '9ihcj8IVW0A=', 'dtype': 'f8'},
               'y0': ' ',
               'yaxis': 'y4'},
              {'alignmentgroup': 'True',
               'boxpoints': 'all',
               'fillcolor': 'rgba(255,255,255,

In [ ]:
import pandas as pd

def flag_maze(df):
    return pd.DataFrame({
        "emzcompk_zero": df["emzcompk"] == 0,
        "emzinitk_zero": df["emzinitk"] == 0,
        "emzerrk_zero": df["emzerrk"] == 0,
        "emzoverk_zero": df["emzoverk"] == 0,
        "emztrlsk_zero": df["emztrlsk"] == 0,
        "emztrlsk_too_low": df["emztrlsk"] <= 2,
        # TODO: add healthy reference range checks for emzerrk, emzoverk, emztrlsk
        # once numeric bounds are defined
    })

def flag_gng(df):
    return pd.DataFrame({
        "g2avrtk_zero": df["g2avrtk"] == 0,
        "g2avrtk_over_max": df["g2avrtk"] > 2.7,
        "g2errk_zero": df["g2errk"] == 0,
        "g2fnk_zero": df["g2fnk"] == 0,
        "g2fnk_over_max": df["g2fnk"] >= 126,
        "g2fpk_zero": df["g2fpk"] == 0,
        "g2fpk_over_max": df["g2fpk"] >= 42,
        "gng_error_sum_mismatch": (df["g2fnk"] + df["g2fpk"]) != df["g2errk"],
    })

def flag_stroop(df):
    return pd.DataFrame({
        "vcrtne_zero": df["vcrtne"] == 0,
        "vcrtne_over_max": df["vcrtne"] >= 30,
        "vi_sco1_negative": df["vi_sco1"] < 0,
    })

def flag_digit_span(df):
    valid_digitsp = df["digitsp"].isin([0, 3, 4, 5, 6, 7, 8, 9])
    return pd.DataFrame({
        "digitot_out_of_range": df["digitot"].notna() & ~df["digitot"].between(0, 13),
        "digitsp_invalid_value": df["digitsp"].notna() & ~valid_digitsp,
        "digitsp_zero_but_digitot_nonzero": (df["digitsp"] == 0) & (df["digitot"] != 0),
        "digitsp_missing_but_digitot_present": df["digitsp"].isna() & df["digitot"].notna(),
    })

# flags_df = pd.concat(
#     [
#         flag_maze(aim3_screen_df_filtered),
#         flag_gng(aim3_screen_df_filtered),
#         flag_stroop(aim3_screen_df_filtered),
#         flag_digit_span(aim3_screen_df_filtered),
#     ],
#     axis=1,
# )

# aim3_screen_df_filtered["flag_reasons"] = flags_df.apply(
#     lambda row: ", ".join(row.index[row]), axis=1
# )
# aim3_screen_df_filtered["has_flag"] = flags_df.any(axis=1)

# aim3_screen_df_filtered.loc[
#     aim3_screen_df_filtered["has_flag"], ["record_id", "study_id", "flag_reasons"]
# ]
